# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed. This cell can be commented if already installed.
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset schema
dataset = mlc.Dataset(url)

# Access the metadata object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data in *record sets*. Each record set aggregates fields and columns. We use the `@id` identifier to explore and reference them.

In [ ]:
# Show all available record sets in the dataset
# `@id` is used to reference entities as required

record_sets = []

# The Croissant metadata contains list of record sets as 'recordSet' attribute.
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name', '(no name)')}, description: {rs.get('description', '(no description)')}")
        record_sets.append(rs['@id'])
else:
    print('No explicit record sets found in metadata.')

In [ ]:
# As an explicit example - enumerate fields within a found record set by `@id`.
if record_sets:
    example_record_set_id = record_sets[0]
    example_record_set = None
    for rs in metadata.recordSet:
        if rs['@id'] == example_record_set_id:
            example_record_set = rs
            break
    if example_record_set and 'field' in example_record_set:
        print(f"Fields for RecordSet {example_record_set_id}:")
        for field in example_record_set['field']:
            print(f"  Field @id: {field['@id']}, name: {field.get('name', '(no name)')}, dataType: {field.get('dataType', '(unknown)')}")
    else:
        print(f"No fields found for RecordSet {example_record_set_id}")
else:
    print('No record sets found to inspect.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below we use the discovered record set and field `@id`s. *You may adjust the list of record sets and the record set chosen for further exploration based on the outcome above.*

In [ ]:
# Extract all records sets in a dictionary of DataFrames, indexed by their @id.
dataframes = {}
if record_sets:
    for record_set_id in record_sets:
        print(f"Loading records from RecordSet @id: {record_set_id}")
        # Each record is a dict where keys are field @id
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for RecordSet {record_set_id}: {df.columns.tolist()}")
    # For demonstration, show first few rows of the first record set
    chosen_record_set_id = record_sets[0]
    chosen_df = dataframes[chosen_record_set_id]
    chosen_df.head()
else:
    print('No record sets available to extract or parse.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data by key attributes.

**Note:** Ensure to replace `<numeric_field_id>` and `<group_field_id>` with the actual `@id` of the relevant field, as printed above.

In [ ]:
# Example: Analyze a numeric field and perform transformations on one record set
import numpy as np

# --- You may need to modify these values based on your previous outputs ---
# For demonstration, we'll infer IDs if possible
if record_sets:
    record_set_id = chosen_record_set_id
    df = dataframes[record_set_id]

    # Try to infer a numeric field (e.g., protein abundance, molecular weight, coverage, etc.)
    # For this dataset, common numeric fields might include e.g. 'cr:field/mw' (molecular weight), 'cr:field/abundance', etc.
    # Let's pick the first column with dtype float or int
    numeric_field_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        print(f"Using numeric field: {numeric_field_id}")

        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} (z-score) for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another field, e.g. by accession or category if they exist
        group_field_candidates = [c for c in df.columns if c != numeric_field_id]
        group_field_id = None
        for col in group_field_candidates:
            if df[col].dtype == object and df[col].nunique() < 10:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id} (showing means):")
            print(grouped_df.head())
        else:
            print('No suitable group field found for grouping operation.')

    else:
        print('No numeric field found in record set for EDA.')
else:
    print('No record set available for EDA step.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Example visualizations include histograms of numeric values, boxplots, or scatter plots of relationships. Adjust `numeric_field_id` and `group_field_id` as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_field_candidates:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If we found a group field
    if group_field_id:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'Boxplot of {numeric_field_id} grouped by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('Insufficient data to plot. Check EDA output for available columns.')

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset using the Croissant specification with `mlcroissant`, inspected its structure using `@id` references, loaded record sets as DataFrames, performed simple exploratory data analysis, and visualized some numeric features.

- Using the Croissant schema allows indexing fields and record sets reliably via their `@id`, ensuring clarity in complex biomedical datasets.
- The dataset is well-suited for proteomics and biomedical signal exploration, with structured fields representing protein properties, abundance, and identities.

For further in-depth analysis, refer to the dataset's official FAIR² package documentation.